<h1> 1. Sample (ex. 10k) </h1>

In [5]:
import pandas as pd
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

# --- CONFIG ---
BASE_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\English")
MASKS_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Clean_Data\cleaning_masks")
SAMPLE_OUTPUT_DIR = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\Data\Sample_Data")
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SIZE = 10_000

def collect_training_samples(target_years):
    all_samples = []
    
    # Calculate target samples per year
    num_years = len(target_years)
    samples_per_year = SAMPLE_SIZE // num_years
    
    print(f"Targeting ~{samples_per_year} rows per year across {num_years} years...")

    for year in tqdm(target_years, desc="Sampling Years"):
        year_path = MASKS_DIR / str(year)
        if not year_path.exists():
            continue
            
        # Find all mask files for this specific year
        year_masks = list(year_path.glob("subset_*_mask.parquet"))
        if not year_masks:
            continue
            
        # Distribute the year's quota evenly across its files
        samples_per_file = max(1, samples_per_year // len(year_masks))
        
        for mask_path in year_masks:
            try:
                # 1. Load the Mask (Clean IDs)
                mask_df = pd.read_parquet(mask_path)
                if mask_df.empty: continue
                
                # 2. Randomly select indices
                n_to_draw = min(len(mask_df), samples_per_file)
                selected_ids = mask_df.sample(n=n_to_draw)['original_index'].tolist()
                
                # 3. Load Raw Data
                raw_filename = mask_path.name.replace("_mask.parquet", ".parquet")
                raw_path = BASE_DIR / str(year) / raw_filename
                
                raw_df = pd.read_parquet(raw_path, columns=['text'])
                sampled_text = raw_df.loc[selected_ids].copy()

                sampled_text['original_index'] = selected_ids
                sampled_text['year'] = year
                sampled_text['source_file'] = raw_filename
                
                all_samples.append(sampled_text)
                
            except Exception as e:
                print(f"Error sampling {mask_path}: {e}")

    # Combine and save
    if not all_samples:
        print("No samples collected. Check your MASKS_DIR paths.")
        return

    # Extract start and end for the filename
    start_year = min(target_years)
    end_year = max(target_years)
    output_path = SAMPLE_OUTPUT_DIR / f"training_samples_{start_year}_{end_year}.parquet"

    # Shuffle and Save
    final_df = pd.concat(all_samples).sample(frac=1).reset_index(drop=True)
    final_df.to_parquet(output_path)

    print(f"Successfully saved {len(final_df)} samples.")

In [6]:
if __name__ == "__main__":
    # 1. Generate the non-overlapping periods
    # Initial custom block: 1678 to 1700
    # all_periods = [range(1678, 1701)] 
    
    # # Subsequent 25-year blocks: 1701-1725, 1726-1750, etc.
    # for start in range(1701, 2002, 25):
    #     end = min(start + 24, 2023)
    #     all_periods.append(range(start, end + 1))

    # all_periods = [range(1851, 1876)]

    all_periods = [
        range(1701, 1726),
        range(1726, 1751),
        range(1751, 1776),
        range(1776, 1801),
        range(1801, 1826),
        range(1826, 1851),
    ]
    # 2. Run the collection for each period
    for period_range in all_periods:
        print(f"\n--- Processing Period: {min(period_range)}_{max(period_range)} ---")
        collect_training_samples(list(period_range))


--- Processing Period: 1701_1725 ---
Targeting ~400 rows per year across 25 years...


Sampling Years:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully saved 10000 samples.

--- Processing Period: 1726_1750 ---
Targeting ~400 rows per year across 25 years...


Sampling Years:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully saved 10000 samples.

--- Processing Period: 1751_1775 ---
Targeting ~400 rows per year across 25 years...


...

Successfully saved 10000 samples.

--- Processing Period: 1801_1825 ---
Targeting ~400 rows per year across 25 years...


Sampling Years:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully saved 10000 samples.

--- Processing Period: 1826_1850 ---
Targeting ~400 rows per year across 25 years...


Sampling Years:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully saved 10000 samples.


In [2]:
all_periods

[range(1678, 1701),
 range(1701, 1726),
 range(1726, 1751),
 range(1751, 1776),
 range(1776, 1801),
 range(1801, 1826),
 range(1826, 1851),
 range(1851, 1876),
 range(1876, 1901),
 range(1901, 1926),
 range(1926, 1951),
 range(1951, 1976),
 range(1976, 2001),
 range(2001, 2024)]